In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# CELL 0 — Shared prep config  (run once; used by ALL 3 datasets)

In [2]:

import os, re, random, hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- shared output contract (identical for every dataset) ----
TARGET_SIZE  = 256                       # materialize letterboxed to 256²; online random-crop to 224² at train time
PAD_COLOR    = (0, 0, 0)                  # letterbox fill
IMG_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
OUT_ROOT     = Path("/kaggle/working/prepared")   # cleaned dataset built here, then published as a Kaggle Dataset
SPLIT_RATIOS = {"train": 0.80, "val": 0.10, "test": 0.10}   # used when a source ships no split (rice)

# the manifest schema every dataset must emit, so the three merge cleanly:
MANIFEST_COLS = ["src_path", "filename", "label", "source_dataset", "split", "group_id"]

INPUT_ROOT = "/kaggle/input"
print("Mounted datasets:", os.listdir(INPUT_ROOT))
print("Output root     :", OUT_ROOT)

Mounted datasets: ['datasets']
Output root     : /kaggle/working/prepared


# CELL W1 — Wheat: locate dataset + build raw manifest

In [3]:
cands = [os.path.join(INPUT_ROOT, d) for d in os.listdir(INPUT_ROOT)]
DATA_ROOT = next((c for c in cands if "wheat" in c.lower()), cands[0])
print("Wheat DATA_ROOT =", DATA_ROOT)

SPLIT_NAMES = {"train":"train","training":"train","val":"val","valid":"val",
               "validation":"val","test":"test","testing":"test"}

rows = []
for p in Path(DATA_ROOT).rglob("*"):
    if p.is_file() and p.suffix.lower() in IMG_EXTS:
        parts = [pp.lower() for pp in p.relative_to(DATA_ROOT).parts]
        split = next((SPLIT_NAMES[pp] for pp in parts if pp in SPLIT_NAMES), "all")
        rows.append({"src_path": str(p), "filename": p.name,
                     "raw_label": p.parent.name, "split": split})

wheat = pd.DataFrame(rows)
print("Total wheat images:", len(wheat))
print("\nSplit counts:\n", wheat["split"].value_counts())
print("\nRaw label count:", wheat["raw_label"].nunique(), "(expect 45 — the corrupted naming)")

Wheat DATA_ROOT = /kaggle/input/datasets
Total wheat images: 14154

Split counts:
 split
train    13104
test       750
val        300
Name: count, dtype: int64

Raw label count: 45 (expect 45 — the corrupted naming)


# CELL W2 — Wheat: repair 45 → 15 labels + add crop prefix

In [4]:
SUFFIXES = ("_test_valid", "_valid", "_test")   # longest first → catches the broken 'blast_test_valid'

def canon_key(raw):
    s = raw.strip().lower().replace(" ", "_")
    for suf in SUFFIXES:
        if s.endswith(suf):
            s = s[:-len(suf)]; break
    return re.sub(r"_+", "_", s).strip("_")

wheat["key"]            = wheat["raw_label"].map(canon_key)
wheat["label"]          = "wheat__" + wheat["key"]      # crop-prefixed canonical label
wheat["source_dataset"] = "wheat_kushagra3204"

n = wheat["key"].nunique()
print(f"Canonical classes: {n}   (expect 15)")
print(sorted(wheat["key"].unique()), "\n")

# per-class × split crosstab + completeness check
ct = pd.crosstab(wheat["key"], wheat["split"])
for s in ["train", "val", "test"]:
    if s not in ct.columns: ct[s] = 0
ct = ct[["train", "val", "test"]]
print(ct)

missing = [k for k in ct.index if (ct.loc[k] == 0).any()]
print("\nClasses missing from a split:", set(missing) if missing else "none ✅")

print("\nDid the broken 'blast_test_valid' fold into 'blast'?")
print(wheat.loc[wheat.raw_label.str.lower() == "blast_test_valid", "key"].value_counts())

Canonical classes: 15   (expect 15)
['aphid', 'black_rust', 'blast', 'brown_rust', 'common_root_rot', 'fusarium_head_blight', 'healthy', 'leaf_blight', 'mildew', 'mite', 'septoria', 'smut', 'stem_fly', 'tan_spot', 'yellow_rust'] 

split                 train  val  test
key                                   
aphid                   903   20    50
black_rust              576   20    50
blast                   647   20    50
brown_rust             1271   20    50
common_root_rot         614   20    50
fusarium_head_blight    611   20    50
healthy                1000   20    50
leaf_blight             842   20    50
mildew                 1081   20    50
mite                    800   20    50
septoria               1144   20    50
smut                   1310   20    50
stem_fly                234   20    50
tan_spot                770   20    50
yellow_rust            1301   20    50

Classes missing from a split: none ✅

Did the broken 'blast_test_valid' fold into 'blast'?
key
blast    2

# CELL W3 — Wheat: compute exact + perceptual hashes
# (slow pass — decodes every image once. Run once, then iterate on W4.)

In [5]:
import hashlib

# perceptual hash (pHash): a 64-bit DCT-based fingerprint. No external libs.
def _dct_matrix(N):
    n = np.arange(N); k = n.reshape(-1, 1)
    M = np.sqrt(2.0 / N) * np.cos(np.pi * (2 * n + 1) * k / (2 * N))
    M[0, :] /= np.sqrt(2.0)
    return M
_D32 = _dct_matrix(32)

def phash64(pil_img):
    g = pil_img.convert("L").resize((32, 32), Image.BILINEAR)
    a = np.asarray(g, dtype=np.float64)
    d = _D32 @ a @ _D32.T                 # 2-D DCT
    flat = d[:8, :8].flatten()            # keep the 8×8 low-frequency block
    med  = np.median(flat[1:])            # median of the 63 non-DC coefficients
    bits = flat > med                     # 64 booleans → 64-bit fingerprint
    h = np.uint64(0)
    for b in bits:
        h = (h << np.uint64(1)) | np.uint64(bool(b))
    return h

file_md5, phash, bad = [], [], []
for i, p in enumerate(wheat["src_path"]):
    try:
        with open(p, "rb") as f:
            file_md5.append(hashlib.md5(f.read()).hexdigest())
        with Image.open(p) as im:
            phash.append(int(phash64(im)))
    except Exception as e:
        file_md5.append(None); phash.append(None); bad.append((p, str(e)))
    if (i + 1) % 2000 == 0:
        print(f"  hashed {i+1}/{len(wheat)}")

wheat["file_md5"] = file_md5
wheat["phash"]    = phash
print("Done. Unreadable images:", len(bad))
print("Exact byte-duplicate files:",
      int(wheat["file_md5"].duplicated(keep=False).sum()),
      "across", wheat["file_md5"].nunique(), "unique blobs")

  hashed 2000/14154
  hashed 4000/14154
  hashed 6000/14154
  hashed 8000/14154
  hashed 10000/14154
  hashed 12000/14154
  hashed 14000/14154
Done. Unreadable images: 0
Exact byte-duplicate files: 4856 across 11143 unique blobs


# CELL W4 — Wheat: near-dup clustering, leakage report, drop exact dups

In [8]:
PHASH_THRESH = 6    # max Hamming distance (of 64 bits) to call two images near-duplicates. Tunable.

valid = wheat.dropna(subset=["phash"]).copy().reset_index()   # 'index' = original row id
H = valid["phash"].to_numpy(dtype=np.uint64)
N = len(H)

# fast popcount on uint64 arrays (SWAR) — no per-pair Python loop
_c1=np.uint64(0x5555555555555555); _c2=np.uint64(0x3333333333333333)
_c4=np.uint64(0x0f0f0f0f0f0f0f0f); _cm=np.uint64(0x0101010101010101)
def popcount64(x):
    x = x - ((x >> np.uint64(1)) & _c1)
    x = (x & _c2) + ((x >> np.uint64(2)) & _c2)
    x = (x + (x >> np.uint64(4))) & _c4
    return (x * _cm >> np.uint64(56)) & np.uint64(0x7f)

# union-find to merge near-dup pairs into transitive groups
parent = list(range(N))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[rb] = ra

BLK = 500                                    # process 500 rows at a time (memory-safe)
for s in range(0, N, BLK):
    e = min(s + BLK, N)
    dist = popcount64(np.bitwise_xor(H[s:e, None], H[None, :]))   # (b, N) Hamming
    for r in range(e - s):
        gi = s + r
        for j in np.where(dist[r] <= PHASH_THRESH)[0]:
            if j > gi: union(gi, int(j))
    if (e % 2500) < BLK: print(f"  clustered {e}/{N}")

valid["group_id"] = [f"wheat_g{find(i):05d}" for i in range(N)]
wheat = wheat.merge(valid.set_index("index")[["group_id"]],
                    left_index=True, right_index=True, how="left")
solo = wheat["group_id"].isna()
wheat.loc[solo, "group_id"] = [f"wheat_solo{i:05d}" for i in range(int(solo.sum()))]

# ---------- DIAGNOSTICS (numbers for your documentation) ----------
g = wheat.groupby("group_id")
sizes = g.size()
print("Near-dup groups (size>1):", int((sizes > 1).sum()), "| largest cluster:", int(sizes.max()))
print("Images inside a multi-image group:", int(wheat["group_id"].map(sizes).gt(1).sum()))

label_conflict = int((g["label"].nunique() > 1).sum())
print("Groups spanning >1 disease label (probable mislabels):", label_conflict)

spl_span   = g["split"].nunique()
leak_groups = spl_span[spl_span > 1].index
leak_imgs   = int(wheat["group_id"].isin(leak_groups).sum())
print("Groups leaking across the SHIPPED split:", len(leak_groups),
      "| images involved:", leak_imgs)

# ---------- ACTION: drop only byte-exact duplicates (keep 1 per md5) ----------
before = len(wheat)
wheat_clean = wheat.sort_values("filename").drop_duplicates("file_md5", keep="first").copy()
print(f"\nDropped {before - len(wheat_clean)} byte-exact duplicate files → {len(wheat_clean)} kept.")
print("Near-dups are KEPT but grouped, so Notebook D can split group-aware.")

  clustered 2500/14154
  clustered 5000/14154
  clustered 7500/14154
  clustered 10000/14154
  clustered 12500/14154
Near-dup groups (size>1): 2299 | largest cluster: 16
Images inside a multi-image group: 6795
Groups spanning >1 disease label (probable mislabels): 329
Groups leaking across the SHIPPED split: 671 | images involved: 2321

Dropped 3011 byte-exact duplicate files → 11143 kept.
Near-dups are KEPT but grouped, so Notebook D can split group-aware.


# CELL W5 — diagnose the 329 label conflicts

In [9]:
# ============================================================
# CELL W5 — Wheat: are the label-conflict groups mislabels or loose merges?
# ============================================================
conf_ids = [gid for gid, sub in wheat.groupby("group_id") if sub["label"].nunique() > 1]

def max_intra_dist(sub):
    hs = sub["phash"].dropna().to_numpy(dtype=np.uint64)
    if len(hs) < 2: return 0
    d = popcount64(np.bitwise_xor(hs[:, None], hs[None, :]))
    return int(d.max())

rows = []
for gid in conf_ids:
    sub = wheat[wheat.group_id == gid]
    rows.append({"group_id": gid, "n": len(sub),
                 "labels": ", ".join(sorted(sub.key.unique())),
                 "max_dist": max_intra_dist(sub)})
cdf = pd.DataFrame(rows)

print("Label-conflict groups:", len(cdf), "\n")
print("Max intra-group Hamming distance distribution:")
print(cdf["max_dist"].value_counts().sort_index())
print("\n  dist 0-2 = near-identical image, different label  -> GENUINE MISLABEL (drop/relabel)")
print("  dist 5-6 = only loosely similar                   -> pHASH FALSE-MERGE (harmless)\n")

print("Most-conflicting pairings (which diseases get confused):")
print(cdf["labels"].value_counts().head(10))

Label-conflict groups: 329 

Max intra-group Hamming distance distribution:
max_dist
0    282
2     31
4      6
6      8
8      2
Name: count, dtype: int64

  dist 0-2 = near-identical image, different label  -> GENUINE MISLABEL (drop/relabel)
  dist 5-6 = only loosely similar                   -> pHASH FALSE-MERGE (harmless)

Most-conflicting pairings (which diseases get confused):
labels
black_rust, brown_rust       109
leaf_blight, septoria         97
leaf_blight, tan_spot         46
brown_rust, leaf_blight        9
common_root_rot, tan_spot      6
blast, leaf_blight             4
aphid, leaf_blight             4
leaf_blight, smut              4
smut, tan_spot                 4
aphid, mite                    4
Name: count, dtype: int64


# CELL W6 — drop confirmed mislabels

In [10]:
# ============================================================
# CELL W6 — Wheat: drop genuine-mislabel groups (intra-dist <= 2)
# ============================================================
MISLABEL_DIST = 2   # only groups this tight are trusted as true same-image conflicts

drop_ids = []
for gid, sub in wheat.groupby("group_id"):
    if sub["label"].nunique() > 1 and max_intra_dist(sub) <= MISLABEL_DIST:
        drop_ids.append(gid)

before = len(wheat_clean)
wheat_clean = wheat_clean[~wheat_clean["group_id"].isin(drop_ids)].copy()
print(f"Mislabel groups dropped : {len(drop_ids)}")
print(f"Images removed          : {before - len(wheat_clean)}")
print(f"Wheat rows remaining    : {len(wheat_clean)}")
print(f"\nClasses still present   : {wheat_clean['key'].nunique()} (want 15)")
print(wheat_clean["key"].value_counts().sort_index())

Mislabel groups dropped : 313
Images removed          : 469
Wheat rows remaining    : 10674

Classes still present   : 15 (want 15)
key
aphid                    859
black_rust               315
blast                    584
brown_rust              1192
common_root_rot          568
fusarium_head_blight     639
healthy                 1036
leaf_blight              567
mildew                  1125
mite                     764
septoria                 349
smut                     504
stem_fly                 172
tan_spot                 649
yellow_rust             1351
Name: count, dtype: int64


# CELL W7 — quality filter + letterbox-resize + materialize

In [11]:
# ============================================================
# CELL W7 — Wheat: near-blank filter, letterbox to 256², write images + manifest
# ============================================================
BLANK_VAR_THRESH = 8.0    # drop only near-featureless images (very low Laplacian variance)

OUT_IMG = OUT_ROOT / "wheat" / "images"
OUT_IMG.mkdir(parents=True, exist_ok=True)

def lap_var(gray):                        # cheap blur/blank score
    a = np.asarray(gray, dtype=np.float64)
    k = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float64)
    from numpy.lib.stride_tricks import sliding_window_view as swv
    w = swv(a, (3,3))
    return float((w * k).sum(axis=(2,3)).var())

records, dropped_blank, n_fail = [], 0, 0
for i, r in enumerate(wheat_clean.itertuples(index=False)):
    try:
        with Image.open(r.src_path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            if lap_var(im.resize((128,128), Image.BILINEAR).convert("L")) < BLANK_VAR_THRESH:
                dropped_blank += 1; continue
            im = ImageOps.pad(im, (TARGET_SIZE, TARGET_SIZE),
                              method=Image.BILINEAR, color=PAD_COLOR)  # aspect-preserving letterbox
            fn = f"{r.key}_{i:05d}.jpg"
            im.save(OUT_IMG / fn, "JPEG", quality=92)
        records.append({"src_path": r.src_path, "filename": fn, "label": r.label,
                        "source_dataset": "wheat_kushagra3204", "split": "unassigned",
                        "group_id": r.group_id})
    except Exception:
        n_fail += 1

man = pd.DataFrame(records, columns=MANIFEST_COLS)
man.to_csv(OUT_ROOT / "wheat" / "manifest.csv", index=False)

print(f"Near-blank dropped : {dropped_blank}")
print(f"Failed to process  : {n_fail}")
print(f"Images written     : {len(man)}  ->  {OUT_IMG}")
print(f"Manifest rows      : {len(man)}   cols: {list(man.columns)}")
print("\nPer-class after materialize:")
print(man['label'].value_counts().sort_index())

Near-blank dropped : 1
Failed to process  : 0
Images written     : 10673  ->  /kaggle/working/prepared/wheat/images
Manifest rows      : 10673   cols: ['src_path', 'filename', 'label', 'source_dataset', 'split', 'group_id']

Per-class after materialize:
label
wheat__aphid                    859
wheat__black_rust               315
wheat__blast                    584
wheat__brown_rust              1191
wheat__common_root_rot          568
wheat__fusarium_head_blight     639
wheat__healthy                 1036
wheat__leaf_blight              567
wheat__mildew                  1125
wheat__mite                     764
wheat__septoria                 349
wheat__smut                     504
wheat__stem_fly                 172
wheat__tan_spot                 649
wheat__yellow_rust             1351
Name: count, dtype: int64


In [16]:
!kaggle datasets list -m -s wheat-cleaned-256

No datasets found
